In [ ]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

In [ ]:
%%writefile src/evaluation.py
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Benchmark:
    """One dataset to evaluate the pipeline against."""
    name: str
    csv_path: Path
    goal: str
    baseline_metric_name: str   # e.g. "roc_auc" -- must match a key extract_metrics could produce
    baseline_value: float       # the score to compare against
    higher_is_better: bool = True

In [ ]:
from sklearn.datasets import load_breast_cancer, load_wine, fetch_california_housing
from src.config import RUNS_DIR
from src.evaluation import Benchmark

BENCH_DIR = RUNS_DIR / "benchmarks"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

# 1. Breast cancer -- binary classification, you've already run this one many times today
bc_path = BENCH_DIR / "breast_cancer.csv"
load_breast_cancer(as_frame=True).frame.to_csv(bc_path, index=False)

# 2. Wine -- 3-class classification, a genuinely different problem shape to test on
wine_df = load_wine(as_frame=True).frame
wine_path = BENCH_DIR / "wine.csv"
wine_df.to_csv(wine_path, index=False)

# 3. California housing -- regression, tests the planner's regression branch for real
housing_path = BENCH_DIR / "california_housing.csv"
fetch_california_housing(as_frame=True).frame.to_csv(housing_path, index=False)

benchmarks = [
    Benchmark(
        name="Breast Cancer (binary classification)",
        csv_path=bc_path,
        goal="Predict whether a tumor is malignant or benign, using the 'target' column.",
        baseline_metric_name="roc_auc",
        baseline_value=0.99,   # well-documented: logistic regression typically reaches ~0.99 ROC-AUC here
    ),
    Benchmark(
        name="Wine (3-class classification)",
        csv_path=wine_path,
        goal="Predict the wine cultivar/class, using the 'target' column.",
        baseline_metric_name="accuracy",
        baseline_value=0.97,   # standard classifiers reach ~97-100% on this small, clean dataset
    ),
    Benchmark(
        name="California Housing (regression)",
        csv_path=housing_path,
        goal="Predict the median house value, using the 'MedHouseVal' column.",
        baseline_metric_name="r2",
        baseline_value=0.80,   # a well-tuned random forest typically reaches R² around 0.80-0.85
    ),
]

for b in benchmarks:
    print(b.name, "->", b.csv_path)

In [ ]:
from pathlib import Path

# List what's actually in the notebooks folder to get the exact filename
notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
print([f.name for f in notebooks_dir.glob("*.csv")])

In [ ]:
import shutil
from pathlib import Path

source = Path("notebooks") / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
dest_dir = Path("data/raw")
dest_dir.mkdir(parents=True, exist_ok=True)
dest = dest_dir / "telco_churn.csv"

shutil.copy(source, dest)
print("Copied to:", dest, "| exists:", dest.exists())

In [ ]:
import pandas as pd
df = pd.read_csv(dest)
print(df.shape)
print(df.columns.tolist())
print(df["Churn"].value_counts() if "Churn" in df.columns else "No 'Churn' column found -- check actual name above")

In [ ]:
from pathlib import Path
from src.evaluation import Benchmark

telco_path = Path("data/raw/telco_churn.csv")   # from where you saved it last step

benchmarks.append(Benchmark(
    name="Telco Customer Churn (binary classification)",
    csv_path=telco_path,
    goal=(
        "Predict whether the customer churned, using the 'Churn' column "
        "(values 'Yes'/'No'). Drop non-predictive identifier columns such as 'customerID'."
    ),
    baseline_metric_name="roc_auc",
    baseline_value=0.837,   # your own hand-built baseline from the earlier RAG/churn project
))

for b in benchmarks:
    print(b.name)

In [ ]:
%%writefile -a src/evaluation.py


def find_best_matching_metric(result, metric_name: str):
    """Search every step's stdout (most recent step first) for a metric whose
    key contains `metric_name` as a substring. Returns (value, source_step_title) or (None, None).
    """
    from src.tracking import extract_metrics

    for log in reversed(result.logs):   # most recent step first -- final metrics usually live there
        if not log.approved:
            continue
        metrics = extract_metrics(log.stdout)
        matches = {k: v for k, v in metrics.items() if metric_name in k}
        if matches:
            # If multiple match (e.g. "train_roc_auc" and "test_roc_auc"), prefer one with "test" in the name
            test_matches = {k: v for k, v in matches.items() if "test" in k or "val" in k}
            chosen = test_matches if test_matches else matches
            best_key = max(chosen, key=lambda k: len(k))  # arbitrary tie-break: most specific name
            return chosen[best_key], log.title
    return None, None

In [ ]:
%%writefile -a src/evaluation.py


def run_evaluation_suite(benchmarks: list, run_pipeline_fn, runs_dir, verbose: bool = True):
    """Run the pipeline against every benchmark and collect results for comparison.

    run_pipeline_fn: pass src.orchestrator.run_pipeline (kept as a parameter so this
                      module doesn't need to import orchestrator directly -- avoids
                      a circular import, since orchestrator doesn't need to know about evaluation).
    """
    from pathlib import Path
    import pickle

    eval_results = []

    for bench in benchmarks:
        if verbose:
            print(f"\n{'=' * 60}\nEvaluating: {bench.name}\n{'=' * 60}")

        workspace = Path(runs_dir) / "eval" / bench.name.split(" (")[0].replace(" ", "_").lower() / "workspace"
        workspace.mkdir(parents=True, exist_ok=True)

        try:
            result = run_pipeline_fn(csv_path=bench.csv_path, goal=bench.goal,
                                     workspace=workspace, verbose=verbose)
        except Exception as e:
            eval_results.append({
                "benchmark": bench.name, "pipeline_success": False,
                "agent_metric": None, "metric_source_step": None,
                "baseline_metric_name": bench.baseline_metric_name,
                "baseline_value": bench.baseline_value, "error": str(e),
            })
            continue

        # Save every run so nothing is lost to a later kernel restart
        with open(workspace / "run_result.pkl", "wb") as f:
            pickle.dump(result, f)

        if not result.success:
            eval_results.append({
                "benchmark": bench.name, "pipeline_success": False,
                "agent_metric": None, "metric_source_step": None,
                "baseline_metric_name": bench.baseline_metric_name,
                "baseline_value": bench.baseline_value,
                "error": f"Failed at step {result.failed_at_step}",
            })
            continue

        agent_value, source_step = find_best_matching_metric(result, bench.baseline_metric_name)
        eval_results.append({
            "benchmark": bench.name, "pipeline_success": True,
            "agent_metric": agent_value, "metric_source_step": source_step,
            "baseline_metric_name": bench.baseline_metric_name,
            "baseline_value": bench.baseline_value, "error": None,
        })

    return eval_results

In [ ]:
from src.orchestrator import run_pipeline
from src.evaluation import run_evaluation_suite
from src.config import RUNS_DIR

# Test with just the first benchmark first
eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)

for r in eval_results:
    print(r)

In [ ]:
from src.orchestrator import run_pipeline
from src.evaluation import run_evaluation_suite
from src.config import RUNS_DIR

eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)

for r in eval_results:
    print(r)

In [ ]:
from src.orchestrator import run_pipeline
from src.evaluation import run_evaluation_suite
from src.config import RUNS_DIR

eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)

for r in eval_results:
    print(r)

In [ ]:
from src.llm import chat
print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))

In [ ]:
import os
print(os.getcwd())

In [ ]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

print("Working directory:", Path.cwd())

In [ ]:
from src.llm import chat
print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))

In [ ]:
from src.orchestrator import run_pipeline
from src.evaluation import run_evaluation_suite
from src.config import RUNS_DIR

eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)
for r in eval_results:
    print(r)

In [ ]:
with open(".env", encoding="utf-8") as f:
    print(f.read())

In [ ]:
eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)
for r in eval_results:
    print(r)

In [ ]:
%%writefile src/llm.py
import os
import time

from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIConnectionError, APITimeoutError, APIStatusError

load_dotenv()

_client = None


def get_client() -> OpenAI:
    global _client
    if _client is None:
        _client = OpenAI(
            base_url=os.environ["LLM_BASE_URL"],
            api_key=os.environ["LLM_API_KEY"],
            timeout=180,
        )
    return _client


def chat(messages: list, temperature: float | None = None, max_retries: int = 5) -> str:
    client = get_client()
    model = os.environ["LLM_MODEL"]

    kwargs = {"model": model, "messages": messages}
    if temperature is not None:
        kwargs["temperature"] = temperature

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content or ""

        except (RateLimitError, APIConnectionError, APITimeoutError, APIStatusError) as e:
            # APIStatusError covers 503 "service unavailable" and similar transient
            # server-side errors that aren't specifically rate limits.
            if attempt == max_retries - 1:
                raise
            wait = min(60, 5 * 2 ** attempt)
            print(f"[llm] {type(e).__name__}: retrying in {wait}s "
                  f"(attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)

In [ ]:
with open("src/llm.py", encoding="utf-8") as f:
    content = f.read()
print("APIStatusError" in content, "| length:", len(content))

In [ ]:
eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)
for r in eval_results:
    print(r)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)   # force a reload, in case a stale value is cached

print("LLM_BASE_URL:", os.environ.get("LLM_BASE_URL"))
print("LLM_MODEL:", os.environ.get("LLM_MODEL"))

In [ ]:
eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)
for r in eval_results:
    print(r)

In [ ]:
from pathlib import Path
for p in Path(".").rglob(".env"):
    print(p.resolve())

In [ ]:
groq_key = ""

env_content = f"""LLM_BASE_URL=https://api.groq.com/openai/v1
LLM_API_KEY={groq_key}
LLM_MODEL=openai/gpt-oss-120b
"""
with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print("Written. Contents now:")
with open(".env", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import os
from src.llm import get_client   # importing this triggers load_dotenv()

print("LLM_BASE_URL:", os.environ.get("LLM_BASE_URL"))
print("LLM_MODEL:", os.environ.get("LLM_MODEL"))

from src.llm import chat
print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))

In [ ]:
eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)
for r in eval_results:
    print(r)

In [ ]:
from src.orchestrator import run_pipeline
from src.evaluation import run_evaluation_suite
from src.config import RUNS_DIR

eval_results = run_evaluation_suite(benchmarks[:1], run_pipeline, RUNS_DIR, verbose=True)

for r in eval_results:
    print(r)